In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

In [28]:
df = pd.read_csv('raw_loan_dataset.csv')
print("Shape:", df.shape)
df.head(7)
print(df.info())
print("Missing values Count:")
df.isnull().sum()

Shape: (103, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None
Missing values Count:


Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64

In [5]:
for col in ['Income', 'LoanAmount']:
    df[col] = df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [16]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
    "Yes": "Yes", "No": "No" 
}

categorical_cols = ['HasCollateral', 'PreviousDefaults', 'Approved']

for col in categorical_cols:
   
    df[col] = df[col].astype(str).str.strip().str.lower().map(yes_no_map)

print(" Check Value Counts After Normalization ")
for col in categorical_cols:
    print(f"\n{col} Value Counts:\n", df[col].value_counts(dropna=False))

 Check Value Counts After Normalization 

HasCollateral Value Counts:
 HasCollateral
No     52
Yes    48
Name: count, dtype: int64

PreviousDefaults Value Counts:
 PreviousDefaults
No     85
Yes    15
Name: count, dtype: int64

Approved Value Counts:
 Approved
Yes    66
No     34
Name: count, dtype: int64


In [15]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
print ("messing vlue")
print(df.isnull().sum())

messing vlue
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [14]:
initial_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
final_rows = len(df)

print("Duplicates Removed")
print(f"Row count before: {initial_rows} | Row count after: {final_rows}")

Duplicates Removed
Row count before: 100 | Row count after: 100


In [18]:
numeric_cols = ['Income', 'CreditScore', 'LoanAmount', 'EmploymentYears']

print("--- Checkpoint 6: Outlier Capping (IQR) ---")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
  
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(f"Capped {col} bounds to: [{lower_bound:.2f}, {upper_bound:.2f}]")

--- Checkpoint 6: Outlier Capping (IQR) ---
Capped Income bounds to: [-23827.88, 167619.12]
Capped CreditScore bounds to: [344.25, 962.25]
Capped LoanAmount bounds to: [-16687.50, 66212.50]
Capped EmploymentYears bounds to: [-11.28, 35.53]


In [19]:
mapping = {'Yes': 1, 'No': 0}
for col in categorical_cols:
    df[col] = df[col].map(mapping)

print("Target Distribution")
print(df['Approved'].value_counts())

Target Distribution
Approved
1    66
0    34
Name: count, dtype: int64


In [21]:
print("Counts:")
print(df['Approved'].value_counts())

print("Proportions:")
print(df['Approved'].value_counts(normalize=True))

Counts:
Approved
1    66
0    34
Name: count, dtype: int64
Proportions:
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [22]:
df['DebtToIncome'] = df['LoanAmount'] / (df['Income'] + 1e-5)
df['IncomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1e-5)

print("Engineered Features")
print(df[['DebtToIncome', 'IncomePerYearEmployed']].head())

Engineered Features
   DebtToIncome  IncomePerYearEmployed
0      0.237111           98917.282570
1      0.116084            6432.129045
2      0.424889            5273.693938
3      0.270783            1468.805984
4      0.278675            3917.955186


In [29]:
binary_cols = ["Approved", "HasCollateral", "PreviousDefaults"]

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

scale_cols = [col for col in numeric_cols if col not in binary_cols]

scaler = RobustScaler()

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("RobustScaler applied successfully!")
print("Scaled columns:", scale_cols)

df.head()

RobustScaler applied successfully!
Scaled columns: ['CreditScore', 'EmploymentYears']


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,-0.597633,-0.912351,25800,Yes,No,No
1,96482,-0.674556,0.195219,11200,Y,No,Yes
2,28478,NaN,-0.569721,12100,No,No,Yes
3,"$25,851",-0.455621,0.402390,7000,Yes,No,Yes
4,38396,-0.656805,-0.219124,10700,No,No,Approved


In [30]:
print("Any Missing Data:", df.isnull().any().any())
print("Target Class Datatype:", df['Approved'].dtype)

#
df.to_csv("clean_loan_dataset.csv", index=False)
print("Saved cleanly to clean_loan_dataset.csv.")

Any Missing Data: True
Target Class Datatype: object
Saved cleanly to clean_loan_dataset.csv.
